# Aula 3 — Ingestão e Persistência de Dados

**Disciplina:** Big Data Processing — MBA Engenharia de Dados (Mackenzie)

**Objetivo:** Pipeline de ingestão multi-formato com arquitetura Medallion (Bronze → Silver → Gold).

---

## Instruções

1. **No Google Colab:** execute primeiro a célula de SETUP abaixo
2. Execute cada célula sequencialmente (Shift+Enter)
3. Observe como dados de 3 parceiros com formatos diferentes são unificados
4. Ao final, resolva o **Desafio** proposto

---

## 0. Setup (Google Colab)

Execute esta célula **apenas se estiver no Google Colab**. Ela instala Java 17, PySpark e baixa os datasets do GitHub.

Se estiver rodando localmente (Docker/Jupyter), pode pular esta célula.

In [ ]:
# === SETUP GOOGLE COLAB ===
# Detecta automaticamente se está no Colab e configura o ambiente
import sys, os

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("🔧 Ambiente Colab detectado. Instalando dependências...")
    # Java 17 (PySpark 4.x exige Java 17+)
    os.system("apt-get install openjdk-17-jdk-headless -qq > /dev/null 2>&1")
    # PySpark
    os.system("pip install pyspark -q")
    # Baixar datasets do repositório
    if not os.path.exists("/content/repo"):
        os.system("git clone https://github.com/AleTavares/Mackenzie_BigDataProcessing.git /content/repo")
    # Configurar JAVA_HOME
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    os.environ["PATH"] = "/usr/lib/jvm/java-17-openjdk-amd64/bin:" + os.environ["PATH"]
    DATA_DIR = "/content/repo/datasets/aula_03"
    OUTPUT_DIR = "/content/output"
    print("✅ Setup do Colab concluído!")
else:
    # Ambiente local (Docker/Jupyter)
    DATA_DIR = "/home/jovyan/work/data/aula_03"
    OUTPUT_DIR = "/home/jovyan/work/output"
    print("💻 Ambiente local detectado.")

print(f"   DATA_DIR:   {DATA_DIR}")
print(f"   OUTPUT_DIR: {OUTPUT_DIR}")

## 1. Configuração

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, current_timestamp, coalesce, to_date,
    sum, count, round, desc, when, input_file_name
)
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, TimestampType

spark = SparkSession.builder \
    .appName("DataFlow-Aula03-Ingestao") \
    .master("local[*]") \
    .config("spark.driver.memory", "2g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

# Paths de saída (camadas do data lake)
PATH_BRONZE = f"{OUTPUT_DIR}/bronze"
PATH_SILVER = f"{OUTPUT_DIR}/silver"
PATH_GOLD = f"{OUTPUT_DIR}/gold"
PATH_QUARENTENA = f"{OUTPUT_DIR}/quarentena"

print(f"SparkSession criada: {spark.version}")

## 2. Ingestão — Parceiro A (CSV legado)

O Parceiro A envia arquivos CSV com encoding ISO-8859-1 e separador `;`.

In [ ]:
# Ler CSV legado com encoding especial e separador ;
df_parceiro_a = spark.read.csv(
    f"{DATA_DIR}/parceiro_a/",
    header=True,
    inferSchema=True,
    sep=";",
    encoding="ISO-8859-1"
)

# Adicionar metadados de rastreabilidade
df_parceiro_a = df_parceiro_a \
    .withColumn("_source", lit("parceiro_a")) \
    .withColumn("_ingestion_ts", current_timestamp()) \
    .withColumn("_file_origin", input_file_name())

print(f"Parceiro A: {df_parceiro_a.count():,} registros")
df_parceiro_a.printSchema()
df_parceiro_a.show(3, truncate=False)

## 3. Ingestão — Parceiro B (JSON API)

O Parceiro B envia dumps de API em JSON (múltiplos arquivos paginados).

In [ ]:
# Ler todos os JSONs do parceiro B
df_parceiro_b = spark.read.json(f"{DATA_DIR}/parceiro_b/")

# Adicionar metadados
df_parceiro_b = df_parceiro_b \
    .withColumn("_source", lit("parceiro_b")) \
    .withColumn("_ingestion_ts", current_timestamp()) \
    .withColumn("_file_origin", input_file_name())

print(f"Parceiro B: {df_parceiro_b.count():,} registros")
df_parceiro_b.printSchema()
df_parceiro_b.show(3, truncate=False)

## 4. Ingestão — Parceiro C (Parquet)

O Parceiro C já envia em formato otimizado (Parquet) com schema bem definido.

In [ ]:
# Ler Parquet do parceiro C
df_parceiro_c = spark.read.parquet(f"{DATA_DIR}/parceiro_c/")

# Adicionar metadados
df_parceiro_c = df_parceiro_c \
    .withColumn("_source", lit("parceiro_c")) \
    .withColumn("_ingestion_ts", current_timestamp()) \
    .withColumn("_file_origin", input_file_name())

print(f"Parceiro C: {df_parceiro_c.count():,} registros")
df_parceiro_c.printSchema()
df_parceiro_c.show(3, truncate=False)

## 5. Camada Bronze — Persistência Raw

Bronze = dados brutos exatamente como chegaram, com metadados de ingestão adicionados.

In [ ]:
# Salvar cada parceiro separadamente na Bronze (particionado por source)
for nome, df in [("parceiro_a", df_parceiro_a), ("parceiro_b", df_parceiro_b), ("parceiro_c", df_parceiro_c)]:
    df.write \
        .mode("overwrite") \
        .partitionBy("_source") \
        .parquet(f"{PATH_BRONZE}/{nome}")
    print(f"  Bronze/{nome}: salvo")

print("\nCamada Bronze concluida!")

In [ ]:
# Verificar a Bronze
df_bronze_a = spark.read.parquet(f"{PATH_BRONZE}/parceiro_a")
df_bronze_b = spark.read.parquet(f"{PATH_BRONZE}/parceiro_b")
df_bronze_c = spark.read.parquet(f"{PATH_BRONZE}/parceiro_c")

print(f"Bronze parceiro_a: {df_bronze_a.count():,}")
print(f"Bronze parceiro_b: {df_bronze_b.count():,}")
print(f"Bronze parceiro_c: {df_bronze_c.count():,}")
print(f"Total Bronze: {df_bronze_a.count() + df_bronze_b.count() + df_bronze_c.count():,}")

## 6. Camada Silver — Normalização

Silver = dados limpos, com schema unificado, tipos corretos e validações aplicadas.

In [ ]:
# Normalizar schemas para uniao
# Selecionar colunas comuns com nomes padronizados
colunas_padrao = [
    "order_id", "customer_id", "product_id",
    "quantity", "unit_price", "total_amount",
    "order_date", "shipping_state", "status",
    "_source", "_ingestion_ts"
]

# Selecionar e padronizar cada fonte (adaptar conforme schema real)
df_norm_a = df_bronze_a.select(
    [col(c) for c in colunas_padrao if c in df_bronze_a.columns]
)
df_norm_b = df_bronze_b.select(
    [col(c) for c in colunas_padrao if c in df_bronze_b.columns]
)
df_norm_c = df_bronze_c.select(
    [col(c) for c in colunas_padrao if c in df_bronze_c.columns]
)

# Union (usa unionByName para lidar com colunas em ordens diferentes)
df_silver_raw = df_norm_a.unionByName(df_norm_b, allowMissingColumns=True) \
    .unionByName(df_norm_c, allowMissingColumns=True)

print(f"Silver (antes de validacao): {df_silver_raw.count():,} registros")
df_silver_raw.printSchema()

## 7. Validações e Quarentena

Dados inválidos são separados com o motivo da rejeição.

In [ ]:
from pyspark.sql.functions import array_remove, array, concat_ws

# Definir regras de validacao
df_validado = df_silver_raw \
    .withColumn("_erro_order_null",
        when(col("order_id").isNull(), lit("order_id_nulo")).otherwise(lit(None).cast("string"))
    ) \
    .withColumn("_erro_valor_negativo",
        when(col("total_amount") <= 0, lit("valor_negativo")).otherwise(lit(None).cast("string"))
    ) \
    .withColumn("_erro_quantidade",
        when(col("quantity") <= 0, lit("quantidade_invalida")).otherwise(lit(None).cast("string"))
    )

# Consolidar motivos de rejeicao
df_validado = df_validado.withColumn(
    "_rejection_reasons",
    concat_ws(", ",
        col("_erro_order_null"),
        col("_erro_valor_negativo"),
        col("_erro_quantidade")
    )
)

# Separar validos e quarentena
df_validos = df_validado.filter(col("_rejection_reasons") == "") \
    .drop("_erro_order_null", "_erro_valor_negativo", "_erro_quantidade", "_rejection_reasons")

df_quarentena = df_validado.filter(col("_rejection_reasons") != "") \
    .withColumn("_quarantine_ts", current_timestamp()) \
    .drop("_erro_order_null", "_erro_valor_negativo", "_erro_quantidade")

# Metricas
total_raw = df_silver_raw.count()
total_validos = df_validos.count()
total_quarentena = df_quarentena.count()

print(f"Total entrada:    {total_raw:,}")
print(f"Validos:          {total_validos:,}")
print(f"Quarentena:       {total_quarentena:,}")
print(f"Taxa rejeicao:    {total_quarentena/total_raw*100:.1f}%")
print(f"Conservacao OK:   {total_validos + total_quarentena == total_raw}")

In [ ]:
# Ver amostras da quarentena
print("=== REGISTROS EM QUARENTENA ===")
df_quarentena.select("order_id", "total_amount", "quantity", "_source", "_rejection_reasons").show(10, truncate=False)

In [ ]:
# Persistir Silver e Quarentena
df_validos.write \
    .mode("overwrite") \
    .partitionBy("shipping_state") \
    .parquet(PATH_SILVER)

df_quarentena.write \
    .mode("overwrite") \
    .parquet(PATH_QUARENTENA)

print("Silver e Quarentena salvos!")

## 8. Camada Gold — Agregações de Negócio

Gold = dados prontos para consumo (relatórios, dashboards, ML).

In [ ]:
# Ler da Silver
df_silver = spark.read.parquet(PATH_SILVER)

# Gold: Faturamento por estado e parceiro
df_gold_faturamento = df_silver \
    .groupBy("shipping_state", "_source") \
    .agg(
        round(sum("total_amount"), 2).alias("faturamento"),
        count("order_id").alias("pedidos"),
        round(sum("total_amount") / count("order_id"), 2).alias("ticket_medio")
    ) \
    .orderBy(desc("faturamento"))

print("=== GOLD: FATURAMENTO POR ESTADO E PARCEIRO ===")
df_gold_faturamento.show(15)

# Persistir Gold
df_gold_faturamento.write \
    .mode("overwrite") \
    .parquet(f"{PATH_GOLD}/faturamento_estado_parceiro")

print("Gold salva!")

## 9. Resumo do Pipeline

In [ ]:
print("=" * 50)
print("RESUMO DO PIPELINE")
print("=" * 50)
print(f"  Raw (3 fontes):  {total_raw:,} registros")
print(f"  Bronze:          {total_raw:,} registros (raw + metadados)")
print(f"  Silver:          {total_validos:,} registros (limpos)")
print(f"  Quarentena:      {total_quarentena:,} registros ({total_quarentena/total_raw*100:.1f}%)")
print(f"  Gold:            agregacoes prontas")
print("=" * 50)
print(f"  Conservacao:     validos + quarentena = {total_validos + total_quarentena:,} == {total_raw:,}")

In [ ]:
spark.stop()
print("SparkSession encerrada.")

---

# DESAFIO

## Adicionar uma Camada Gold com Análise Temporal

Usando o pipeline acima como base, implemente:

### Requisitos:

1. **Crie uma segunda tabela Gold** com faturamento mensal por parceiro (extraia mês/ano da `order_date`)
2. **Implemente deduplicação** na Silver usando `dropDuplicates([\"order_id\"])` e mostre quantos registros duplicados foram removidos
3. **Adicione um novo check de validação**: datas futuras (order_date > hoje) devem ir para quarentena
4. **Compare performance de formatos**: meça tempo de leitura do mesmo dado em CSV vs Parquet vs JSON
5. **Documente** com print: quantos registros em cada etapa e % de perda

### Dicas:
- Use `month(col(\"order_date\"))` e `year(col(\"order_date\"))` para extrair componentes
- Use `current_date()` para comparar com datas futuras
- Use `time.time()` para medir leitura de cada formato

### Bonus:
- Implemente schema evolution: adicione uma coluna nova em uma fonte e use `mergeSchema=True`
- Crie um dashboard de qualidade com % de rejeição por regra e por fonte

---

**Boa sorte!** Este pipeline é a base exata do que vocês farão no Projeto Final.

In [ ]:
# ===========================================================
# SEU CODIGO DO DESAFIO AQUI
# ===========================================================

